# Data Sythesis

Generates two datasets for a Return Risk Manager:

1. orders.csv        -> order/return-level data for the Return-Risk Scorer
2. customers.csv      -> customer-level data, including shared-identifier fields (device / address / payment fingerprint) for the Abuse-Ring Sentinel

**IMPORTANT**
This is SYNTHETIC data. No real merchant, customer, or transaction data is used anywhere in this project. Ground-truth labels come from a known generative rule (defined below), which lets us honestly report precision/ recall against something, but it also means the numbers describe how well the model recovers MY rule, not real-world fraud. I'm explicit about this and the rule is designed to be realistic (grounded in publicly documented return-fraud patterns such as wardrobing and bracketing) but imperfectly separable, i.e. we inject noise so that a perfect classifier is impossible, which is what real fraud data looks like.

Customer behavior types (hidden from the model, used only to drive generation and for our own auditing):
- normal              (70%): occasional genuine returns
- occasional_returner  (18%): returns more often, still genuine
- serial_abuser         (7%): wardrobing / bracketing patterns
- ring_member           (5%): shares device/address/payment fingerprint with other "distinct" accounts

Two important design choices to avoid common ML mistakes:
1. NO LEAKAGE: rolling features (customer's historical return rate, order count, chargeback count) are computed using only orders that happened strictly BEFORE the current one, per customer, in time order.
2. Hidden 'behavior_type' is NEVER written into orders.csv as a feature -- only into customers.csv for our own auditing / ring-detector evaluation. Using it as a model feature would be cheating (it's essentially the label generator).

In [1]:
import numpy as np
import pandas as pd
import hashlib
import uuid
from datetime import datetime, timedelta

In [2]:
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

N_CUSTOMERS = 6000
SIM_DAYS = 365
START_DATE = datetime(2025, 1, 1)

CATEGORIES = ["apparel", "footwear", "electronics", "mobile", "beauty", "home", "accessories"]
CATEGORY_BASE_RETURN_RATE = {
    "apparel": 0.30, "footwear": 0.26, "accessories": 0.18,
    "beauty": 0.10, "home": 0.14, "electronics": 0.12, "mobile": 0.09,
}
CATEGORY_AVG_VALUE = {
    "apparel": 1400, "footwear": 2200, "accessories": 900,
    "beauty": 700, "home": 1800, "electronics": 6500, "mobile": 14000,
}
PAYMENT_METHODS = ["UPI", "card", "COD", "wallet"]
RETURN_REASONS = ["size_issue", "changed_mind", "damaged", "not_as_described", "wrong_item", "no_longer_needed"]

BEHAVIOR_TYPES = ["normal", "occasional_returner", "serial_abuser", "ring_member"]
BEHAVIOR_WEIGHTS = [0.70, 0.18, 0.07, 0.05]

## Fingerpirnt helper
Produces stable pseudo-hash standing in for a device/address/payment fingerprint.

In [3]:
def make_fingerprint(seed_str: str) -> str:
    return hashlib.sha1(seed_str.encode()).hexdigest()[:12]

## Customer Synth
Generates a synthetic customer dataset designed to simulate both normal customer behavior and coordinated fraudulent activity.
1) Genuine benign sharing (confounder): e.g. families / hostel mates sharing an address. This is NOT abuse -- the ring detector must learn to tell this apart from real abuse rings.
2) Abuse rings: ring_member customers are grouped into rings of 3-9 that share device AND payment fingerprint (a much stronger, more suspicious signal than just a shared address), simulating one person operating several "distinct" accounts.

In [4]:
def build_customers(n: int) -> pd.DataFrame:
    behavior_types = rng.choice(BEHAVIOR_TYPES, size=n, p=BEHAVIOR_WEIGHTS)
    account_age_days = rng.integers(5, 1500, size=n)

    customer_ids = [f"CUST{100000+i}" for i in range(n)]

    device_fp = [make_fingerprint(f"dev-{cid}") for cid in customer_ids]
    address_fp = [make_fingerprint(f"addr-{cid}") for cid in customer_ids]
    payment_fp = [make_fingerprint(f"pay-{cid}") for cid in customer_ids]

    df = pd.DataFrame({
        "customer_id": customer_ids,
        "behavior_type": behavior_types,           # HIDDEN from model features
        "account_age_days": account_age_days,
        "device_fingerprint": device_fp,
        "address_fingerprint": address_fp,
        "payment_fingerprint": payment_fp,
        "ring_id": [None] * n,                      # filled in below
    })

    # --- Build shared-identifier structures ---
    # 1) Genuine benign sharing (confounder): e.g. families / hostel mates
    benign_pool = df.sample(frac=0.06, random_state=1).index.tolist()
    rng.shuffle(benign_pool)
    bi = 0
    benign_group_num = 0
    while bi < len(benign_pool) - 1:
        group_size = rng.integers(2, 4)  # 2-3 people sharing an address
        group = benign_pool[bi: bi + group_size]
        if len(group) < 2:
            break
        shared_addr = make_fingerprint(f"benign-addr-{benign_group_num}")
        df.loc[group, "address_fingerprint"] = shared_addr
        bi += group_size
        benign_group_num += 1

    # 2) Abuse rings: ring_member share device AND payment fingerprint 
    ring_members = df.index[df["behavior_type"] == "ring_member"].tolist()
    rng.shuffle(ring_members)
    ri = 0
    ring_num = 0
    while ri < len(ring_members) - 1:
        group_size = min(rng.integers(3, 10), len(ring_members) - ri)
        if group_size < 3:
            break
        group = ring_members[ri: ri + group_size]
        shared_dev = make_fingerprint(f"ring-dev-{ring_num}")
        shared_pay = make_fingerprint(f"ring-pay-{ring_num}")
        df.loc[group, "device_fingerprint"] = shared_dev
        df.loc[group, "payment_fingerprint"] = shared_pay
        df.loc[group, "ring_id"] = f"RING{ring_num}"
        ri += group_size
        ring_num += 1

    return df

## Order Synth

Generates a synthetic order-level dataset to simulate realistic purchasing, return, and fraudulent return behavior across customers.

1) Normal Order Behavior: Generates orders with realistic variation in category, order value, payment method, delivery time, order time, and weekend activity, with order volume loosely dependent on account age.
2) Historical Behavioral Features: Computes pre-order statistics such as previous order count, return rate, abusive return rate, and chargebacks, ensuring that only past information is used and preventing data leakage.
3) Return Behavior: Simulates whether an order is returned based on the product category and customer's behavioral type, with higher return probabilities for occasional returners, serial abusers, and ring members.
4) Abusive Return Generation: Assigns abuse labels using multiple behavioral signals such as high-value purchases with vague return reasons, deadline-near returns, repeated abusive behavior, and suspicious COD returns, with added noise to ensure imperfect separability.
5) Chargeback Simulation: A subset of abusive returns is escalated to chargebacks, creating an additional indicator of severe fraudulent behavior.

In [5]:
def simulate_orders(customers: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, cust in customers.iterrows():
        btype = cust["behavior_type"]

        # Order volume depends loosely on account age.
        n_orders = max(1, int(rng.poisson(lam=max(1, cust["account_age_days"] / 90))))
        n_orders = min(n_orders, 40)

        order_days = np.sort(rng.integers(0, SIM_DAYS, size=n_orders))

        # Rolling state used to compute pre-order historical features (no leakage).
        past_orders = 0
        past_returns = 0
        past_abusive_returns = 0
        past_chargebacks = 0

        for od in order_days:
            order_date = START_DATE + timedelta(days=int(od))
            category = rng.choice(CATEGORIES)
            base_val = CATEGORY_AVG_VALUE[category]
            order_value = float(np.round(rng.lognormal(mean=np.log(base_val), sigma=0.5), 2))
            payment_method = rng.choice(PAYMENT_METHODS, p=[0.42, 0.28, 0.20, 0.10])
            delivery_days = int(rng.integers(1, 8))
            order_hour = int(rng.integers(0, 24))
            is_weekend = int(order_date.weekday() >= 5)

            # --- historical features computed BEFORE this order is resolved ---
            hist_return_rate = past_returns / past_orders if past_orders > 0 else 0.0
            hist_abusive_rate = past_abusive_returns / past_orders if past_orders > 0 else 0.0
            hist_orders = past_orders
            hist_chargebacks = past_chargebacks
            price_vs_cat_avg = order_value / base_val

            # --- return probability depends on category + behavior type ---
            base_rate = CATEGORY_BASE_RETURN_RATE[category]
            type_multiplier = {
                "normal": 1.0, "occasional_returner": 1.6,
                "serial_abuser": 2.4, "ring_member": 1.4,
            }[btype]
            return_prob = min(0.95, base_rate * type_multiplier)
            is_returned = rng.random() < return_prob

            return_reason = None
            days_to_return = None
            is_abusive = 0

            if is_returned:
                days_to_return = int(rng.integers(1, 31))

                # reason distribution shifts by behavior type
                if btype == "serial_abuser":
                    reason_probs = [0.30, 0.35, 0.10, 0.15, 0.05, 0.05]
                elif btype == "ring_member":
                    reason_probs = [0.15, 0.20, 0.20, 0.25, 0.10, 0.10]
                else:
                    reason_probs = [0.22, 0.28, 0.20, 0.10, 0.10, 0.10]
                return_reason = rng.choice(RETURN_REASONS, p=reason_probs)

                # --- ground-truth abuse rule (documented, imperfectly separable) ---
                abuse_score = 0.0
                if btype == "serial_abuser":
                    abuse_score += 0.55
                if btype == "ring_member":
                    abuse_score += 0.30
                if return_reason in ("changed_mind", "size_issue") and price_vs_cat_avg > 1.3:
                    abuse_score += 0.20          # wardrobing signature: pricier item, vague reason
                if days_to_return >= 25:
                    abuse_score += 0.15          # returned right at the deadline
                if hist_abusive_rate > 0.3:
                    abuse_score += 0.20          # repeat offender
                if payment_method == "COD" and return_reason == "not_as_described":
                    abuse_score += 0.10          # correlates with claim-without-return fraud patterns
                abuse_score += rng.normal(0, 0.18)  # noise -> imperfect separability, by design

                is_abusive = int(abuse_score > 0.55)
                if is_abusive:
                    past_abusive_returns += 1
                    if rng.random() < 0.25:
                        past_chargebacks += 1  # some abusive returns escalate to a chargeback later

                past_returns += 1

            past_orders += 1

            rows.append({
                "order_id": f"ORD{uuid.uuid4().hex[:10]}",
                "customer_id": cust["customer_id"],
                "order_date": order_date.strftime("%Y-%m-%d"),
                "category": category,
                "order_value": order_value,
                "payment_method": payment_method,
                "delivery_days": delivery_days,
                "order_hour": order_hour,
                "is_weekend": is_weekend,
                "account_age_days_at_order": max(0, cust["account_age_days"] - (SIM_DAYS - od)),
                "hist_orders_before": hist_orders,
                "hist_return_rate_before": round(hist_return_rate, 4),
                "hist_abusive_return_rate_before": round(hist_abusive_rate, 4),
                "hist_chargebacks_before": hist_chargebacks,
                "price_vs_category_avg": round(price_vs_cat_avg, 3),
                "is_returned": int(is_returned),
                "return_reason": return_reason,
                "days_to_return": days_to_return,
                "device_fingerprint": cust["device_fingerprint"],
                "address_fingerprint": cust["address_fingerprint"],
                "payment_fingerprint": cust["payment_fingerprint"],
                # label -- only meaningful / used for training on rows where is_returned == 1
                "is_abusive_return": is_abusive,
                # audit-only column, NEVER a model feature:
                "_behavior_type": btype,
            })

    return pd.DataFrame(rows)

## Synthesized data

In [6]:
customers = build_customers(N_CUSTOMERS)
orders = simulate_orders(customers)

In [7]:
customers.head()

,customer_id,behavior_type,account_age_days,device_fingerprint,address_fingerprint,payment_fingerprint,ring_id
0,CUST100000,occasional_returner,764,53d1f543b808,75ca1918fe35,7c2616eed605,None
1,CUST100001,normal,685,56e3efd59587,2568dfb34d5e,6fc1842a8ed9,None
2,CUST100002,occasional_returner,113,2930e5ca5cb6,37c6ac2e61e9,ec4cbf090273,None
3,CUST100003,normal,182,b32d920217ed,cbc5e98fcc2d,3580c31d5c9e,None
4,CUST100004,normal,894,84c81c1d8bc9,e5207270ae88,53a745dbd2b6,None


In [8]:
orders.head()

,order_id,customer_id,order_date,category,order_value,payment_method,delivery_days,order_hour,is_weekend,account_age_days_at_order,...,hist_chargebacks_before,price_vs_category_avg,is_returned,return_reason,days_to_return,device_fingerprint,address_fingerprint,payment_fingerprint,is_abusive_return,_behavior_type
0,ORDbbe512d543,CUST100000,2025-01-24,accessories,1755.70,COD,2,8,0,422,...,0,1.951,0,None,NaN,53d1f543b808,75ca1918fe35,7c2616eed605,0,occasional_returner
1,ORD4b7ac6e534,CUST100000,2025-01-27,home,1637.30,COD,7,3,0,425,...,0,0.910,0,None,NaN,53d1f543b808,75ca1918fe35,7c2616eed605,0,occasional_returner
2,ORD2f144a536f,CUST100000,2025-02-13,mobile,9980.70,card,1,22,0,442,...,0,0.713,0,None,NaN,53d1f543b808,75ca1918fe35,7c2616eed605,0,occasional_returner
3,ORDc71e0d6f80,CUST100000,2025-03-23,mobile,7692.17,UPI,5,17,1,480,...,0,0.549,0,None,NaN,53d1f543b808,75ca1918fe35,7c2616eed605,0,occasional_returner
4,ORDcba89183aa,CUST100000,2025-03-26,footwear,3445.79,card,4,23,0,483,...,0,1.566,1,no_longer_needed,12.0,53d1f543b808,75ca1918fe35,7c2616eed605,0,occasional_returner


In [9]:
print(customers.columns)
print()
print(orders.columns)

Index(['customer_id', 'behavior_type', 'account_age_days',
       'device_fingerprint', 'address_fingerprint', 'payment_fingerprint',
       'ring_id'],
      dtype='object')

Index(['order_id', 'customer_id', 'order_date', 'category', 'order_value',
       'payment_method', 'delivery_days', 'order_hour', 'is_weekend',
       'account_age_days_at_order', 'hist_orders_before',
       'hist_return_rate_before', 'hist_abusive_return_rate_before',
       'hist_chargebacks_before', 'price_vs_category_avg', 'is_returned',
       'return_reason', 'days_to_return', 'device_fingerprint',
       'address_fingerprint', 'payment_fingerprint', 'is_abusive_return',
       '_behavior_type'],
      dtype='object')


In [10]:
returned = orders[orders["is_returned"] == 1]
print(f"customers: {len(customers)}")
print(f"orders:    {len(orders)}")
print(f"returns:   {len(returned)}  ({len(returned)/len(orders):.1%} of orders)")
print(f"abusive returns: {returned['is_abusive_return'].sum()}  "f"({returned['is_abusive_return'].mean():.1%} of returns)")
print(f"ring members: {(customers['behavior_type']=='ring_member').sum()}  " f"across {customers['ring_id'].nunique()} rings")
print("\nBehavior type breakdown:")
print(customers["behavior_type"].value_counts())

customers: 6000
orders:    50149
returns:   10283  (20.5% of orders)
abusive returns: 1175  (11.4% of returns)
ring members: 287  across 51 rings

Behavior type breakdown:
normal                 4245
occasional_returner    1071
serial_abuser           397
ring_member             287
Name: behavior_type, dtype: int64
